# Lab 2 — A CNN from scratch on CIFAR-10

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kleinric/cv-labs/blob/main/lab-02.ipynb)

**COMS4036A / COMS7050A Computer Vision · Week 2**

**Due: Monday 10 August 2026, 09:00.** Groups of up to three; one member submits the group's notebook (`.ipynb`) on Moodle. Every member must be able to explain every cell.

Companion reading is [Chapter 2 of the course book](https://courses.ms.wits.ac.za/~richard/cv/book/chapters/02-learning-the-filters.html). The network you build here is *the* Lab 2 network — the one whose training curves, filter snapshots, and activation atlas appear in the chapter's figures — so you can check everything you produce against the book.

Three pieces of practitioner craft are this week's skills-ladder rung, and all three are marked: **every run logged to Weights & Biases**, **everything seeded**, and the **overfit-one-batch ritual** before any real training.

The Friday session covers Sections 0–4; Sections 5–7 are the take-home half.

## 0. GPU and W&B

Open this notebook in [Colab](https://colab.research.google.com) — the badge above, or upload it as in Lab 1 — and **File ▸ Save a copy in Drive**. Two things are different this week.

**The GPU.** Under **Runtime ▸ Change runtime type**, select a GPU (a T4 is fine). Training this lab's network on the CPU takes over an hour; on the T4 it is minutes. The cell below is Lab 1's runtime check — this time it should print a GPU.

**Weights & Biases.** W&B records every run's configuration, curves, and outputs on a page you can share; from this lab to the end of the course, unlogged runs don't count. Before the login cell, each group member (not one per group — each of you):

1. Create a free account at [wandb.ai](https://wandb.ai) **with your Wits email** — or, if you already have an account, add your Wits email to it under your account settings.
2. Apply for the free **academic plan** from that account ([how and why](https://docs.wandb.ai/support/academic_plan_student/)) — it upgrades your account at no cost, and the course's W&B team, which later labs log to, requires it.
3. When the course's W&B team is ready, a Moodle form will collect your W&B username — submit it and accept the emailed invite. This lab still logs to your personal `cv-lab2` project.

The login cell will ask for the API key from [wandb.ai/authorize](https://wandb.ai/authorize).

In [1]:
# Group members — fill in before submitting.
MEMBERS = [
    # ("Student name", "Student number"),
]
for name, number in MEMBERS:
    print(f"{number}  {name}")

In [2]:
!nvidia-smi

Sun Aug  9 16:56:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 2539199 (2539199-wits) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 1. Setup, and seeding everything

In [4]:
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV)

device: cuda


A training run touches four random number generators: Python's, NumPy's, PyTorch's CPU one, and PyTorch's GPU one. Seed all four, every time, in one function — this is what makes a run *reproducible*: the same seed rebuilds the same setup, and the numbers come back close. (Bitwise-identical is more than GPU arithmetic promises; close is what you should expect.)

In [5]:
def seed_everything(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(0)

CIFAR-10: 50,000 training and 10,000 test images, $32 \times 32$, ten classes. The test set serves as this lab's held-out validation split — that is what the `val/` metrics below are. The transforms normalise each channel with the dataset's own statistics; the training set additionally gets the chapter's light augmentation — random crops and horizontal flips.

In [6]:
norm = transforms.Normalize((0.4914, 0.4822, 0.4465),
                            (0.2470, 0.2435, 0.2616))
train_tf = transforms.Compose([transforms.RandomCrop(32, padding=4),
                               transforms.RandomHorizontalFlip(),
                               transforms.ToTensor(), norm])
test_tf = transforms.Compose([transforms.ToTensor(), norm])

train_set = datasets.CIFAR10("data", train=True, download=True, transform=train_tf)
test_set = datasets.CIFAR10("data", train=False, download=True, transform=test_tf)
# num_workers=0: every random draw — shuffling and augmentation — then
# flows from the seed, which the Q1.2 experiment depends on.
train_loader = torch.utils.data.DataLoader(train_set, batch_size=128,
                                           shuffle=True, num_workers=0)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=512,
                                          num_workers=0)
print(train_set.classes)

100%|██████████| 170M/170M [27:12<00:00, 104kB/s]


['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


**Q1.1.** Display a grid of one training image per class, titled with its class name. (`train_set.data` holds the raw `uint8` images and `train_set.targets` the labels — no un-normalising needed if you read from there.)

In [ ]:
# Q1.1 - one training image per class.
targets = np.array(train_set.targets)
fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for cls, ax in enumerate(axes.flat):
    idx = np.where(targets == cls)[0][0]     # first image with this label
    ax.imshow(train_set.data[idx])           # raw uint8 HxWxC -> no un-normalising
    ax.set_title(train_set.classes[cls])
    ax.axis("off")
fig.suptitle("One training image per class")
plt.tight_layout()
plt.show()

**Q1.2.** Now look at what the network actually sees. Take one batch from `train_loader` and display its first eight images (un-normalise for display: multiply by the std and add the mean, channel-wise, then clip). Run the cell twice — the crops and flips differ, because augmentation is random. Now **Runtime ▸ Restart session and run all**: the same augmented batch comes back. Why different within a session but identical across restarts — which line guarantees it?

In [ ]:
# Q1.2 - what the network actually sees (one augmented batch, un-normalised).
# NOTE: do NOT re-seed here. We want the batch to differ when this cell is run
# twice within a session (the RNG has advanced), yet be identical after a
# "Restart session and run all" (seed_everything(0) in Section 1 resets it).
mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
std  = torch.tensor([0.2470, 0.2435, 0.2616]).view(3, 1, 1)

xb, yb = next(iter(train_loader))            # one batch, crops + flips applied
fig, axes = plt.subplots(1, 8, figsize=(14, 2.2))
for i, ax in enumerate(axes):
    img = xb[i] * std + mean                 # un-normalise, channel-wise
    img = img.clamp(0, 1).permute(1, 2, 0)   # CHW -> HWC for imshow
    ax.imshow(img.numpy())
    ax.set_title(train_set.classes[yb[i]], fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

*Answer:* Within one session, each `next(iter(train_loader))` draws fresh from the global RNGs - shuffling picks which images form the batch, then `RandomCrop`/`RandomHorizontalFlip` draw the augmentation - and every draw advances that state, so calling the cell a second time yields a different batch. Across a *Restart session and run all* the state is reset to the same point, so the first batch reproduces exactly. The line that guarantees it is `seed_everything(0)` in Section 1, together with `num_workers=0` on the loader (with worker subprocesses the augmentation randomness would live outside that seed).

## 2. The MLP baseline

The obvious network flattens the image and connects everything to everything. Build it, count it, train it — it is the baseline the CNN has to beat.

**Q2.1.** Implement `MLP`: flatten, a hidden layer of 512 units with ReLU, then a linear layer to 10 classes. Implement `count_params(model)` returning the total number of trainable parameters, and report the MLP's count. Verify the first layer's share by hand — $3072 \times 512$ weights plus 512 biases, the chapter's flatten-and-connect counting at this width.

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(3 * 32 * 32, 512)   # 3072 -> 512
        self.fc2 = nn.Linear(512, 10)            # 512 -> 10 classes

    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        return self.fc2(x)                       # logits (loss is on the logits)


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print("MLP parameters:", count_params(MLP()))
# First layer by hand: 3072*512 weights + 512 biases
print("fc1 by hand   :", 3072 * 512 + 512)

**Q2.2.** Write the two functions every experiment in this course reuses — and log to W&B from the start. The loss, here and everywhere in this lab, is cross-entropy on the logits: `F.cross_entropy(model(x), y)`.

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    """Return (mean loss, accuracy) over a loader, without touching gradients."""
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEV), y.to(DEV)
        logits = model(x)
        total_loss += F.cross_entropy(logits, y, reduction="sum").item()
        correct += (logits.argmax(1) == y).sum().item()
        n += y.size(0)
    return total_loss / n, correct / n


def train(model, epochs, opt, run_name, config):
    """Train on train_loader, evaluate each epoch, log everything to W&B.

    Returns history as a dict of lists under train/loss, train/acc,
    val/loss, val/acc."""
    model.to(DEV)
    wandb.init(project="cv-lab2", name=run_name, config=config)
    history = {"train/loss": [], "train/acc": [], "val/loss": [], "val/acc": []}
    for epoch in range(epochs):
        model.train()
        run_loss, correct, n = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(DEV), y.to(DEV)
            opt.zero_grad()
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            loss.backward()
            opt.step()
            run_loss += loss.item() * y.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            n += y.size(0)
        tr_loss, tr_acc = run_loss / n, correct / n
        va_loss, va_acc = evaluate(model, test_loader)
        wandb.log({"train/loss": tr_loss, "train/acc": tr_acc,
                   "val/loss": va_loss, "val/acc": va_acc}, step=epoch)
        for k, v in zip(history, (tr_loss, tr_acc, va_loss, va_acc)):
            history[k].append(v)
        print(f"epoch {epoch + 1:2d}/{epochs}  "
              f"train loss {tr_loss:.3f} acc {tr_acc:.3f}  "
              f"val loss {va_loss:.3f} acc {va_acc:.3f}")
    wandb.finish()
    return history

**Q2.3.** Train the MLP for 5 epochs with Adam (`torch.optim.Adam`, learning rate $10^{-3}$) — seed first, and give the run a name and a config recording at least the architecture, optimiser, learning rate, epochs, parameter count, and seed. Report the final test accuracy.

In [ ]:
seed_everything(0)
mlp = MLP().to(DEV)
opt = torch.optim.Adam(mlp.parameters(), lr=1e-3)
config = dict(arch="MLP-512", optimizer="Adam", lr=1e-3, epochs=5,
              params=count_params(mlp), seed=0)
mlp_hist = train(mlp, epochs=5, opt=opt, run_name="mlp-baseline", config=config)
print("final MLP test accuracy:", round(mlp_hist["val/acc"][-1], 4))

## 3. The chapter's network

The template, at lab scale: two blocks of two $3 \times 3$ convolutions each (32 channels, then 64), a $2 \times 2$ max-pool after each block, global average pooling, one linear layer to the classes. No batch norm — that arrives with Chapter 3.

**Q3.1.** Implement it exactly. Every convolution has `padding=1` (so the $3 \times 3$s keep the spatial size — check with the output-size formula) and a ReLU after it. Name the layers `c1`, `c2`, `c3`, `c4`, and `fc`, as in the book's training code — a later question reads `model.c1.weight` by that name. Give `__init__` a `first_kernel=3` argument that sets `c1`'s kernel size (with padding `first_kernel // 2`); Section 6 uses it.

In [ ]:
class LabNet(nn.Module):
    """[conv32-conv32-pool]-[conv64-conv64-pool]-GAP-linear."""

    def __init__(self, first_kernel=3, num_classes=10):
        super().__init__()
        self.c1 = nn.Conv2d(3, 32, first_kernel, padding=first_kernel // 2)
        self.c2 = nn.Conv2d(32, 32, 3, padding=1)
        self.c3 = nn.Conv2d(32, 64, 3, padding=1)
        self.c4 = nn.Conv2d(64, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = F.relu(self.c1(x))
        x = F.relu(self.c2(x))
        x = self.pool(x)              # 32x32 -> 16x16
        x = F.relu(self.c3(x))
        x = F.relu(self.c4(x))
        x = self.pool(x)              # 16x16 -> 8x8
        x = x.mean(dim=(2, 3))        # global average pooling -> (N, 64)
        return self.fc(x)             # logits

**Q3.2.** Before running `count_params`: walk the shapes on paper with the output-size formula, layer by layer, and compute the parameter count of each layer by hand ($C_{\text{out}} \times (C_{\text{in}} \times k \times k + 1)$). Then check both against the code — `count_params(LabNet())` should agree with your sum exactly. How many MLP parameters buy one LabNet?

In [ ]:
print("LabNet parameters:", count_params(LabNet()))

# By hand, C_out * (C_in * k * k + 1):
c1 = 32 * (3 * 3 * 3 + 1)      # 896
c2 = 32 * (32 * 3 * 3 + 1)     # 9,248
c3 = 64 * (32 * 3 * 3 + 1)     # 18,496
c4 = 64 * (64 * 3 * 3 + 1)     # 36,928
fc = 10 * (64 + 1)             # 650
print("by hand      :", c1, c2, c3, c4, fc, "-> sum", c1 + c2 + c3 + c4 + fc)
print("MLP / LabNet :", round(count_params(MLP()) / count_params(LabNet()), 2))

*Answer:* LabNet has **66,218** trainable parameters (896 + 9,248 + 18,496 + 36,928 + 650), matching `count_params` exactly. The MLP has **1,578,506** parameters - about **24x** more, so one MLP buys roughly 24 LabNets. Almost all of that MLP weight sits in its first fully-connected layer (3072 x 512, about 1.57M) spent wiring every pixel to every hidden unit, whereas the CNN shares a handful of tiny kernels across all spatial positions.

## 4. The ritual: overfit one batch

Before any real training run, always: take a single batch and drive the training loss to zero on it. A network that cannot memorise 128 images has a wiring bug — the loss is not connected to what you think, the labels are shuffled, the learning rate is absurd — and no amount of patience on the full set will fix it. Thirty seconds now saves an hour later.

**Q4.1.** Seed, then take one batch from `train_loader` and train a fresh `LabNet` on that batch alone — several hundred Adam steps; 800 is plenty — printing the loss as it falls. Log this run to W&B too (name it so it is obviously the ritual, not a result). How many steps until the loss is below 0.01?

In [ ]:
# Q4.1 - overfit one batch: drive the loss to ~0 on a single batch.
seed_everything(0)
net = LabNet().to(DEV)
xb, yb = next(iter(train_loader))
xb, yb = xb.to(DEV), yb.to(DEV)
opt = torch.optim.Adam(net.parameters(), lr=1e-3)

wandb.init(project="cv-lab2", name="ritual-overfit-one-batch",
           config=dict(arch="LabNet", optimizer="Adam", lr=1e-3, steps=800,
                       batch=xb.size(0), params=count_params(net), seed=0))
net.train()
hit = None
for step in range(800):
    opt.zero_grad()
    loss = F.cross_entropy(net(xb), yb)
    loss.backward()
    opt.step()
    wandb.log({"train/loss": loss.item()}, step=step)
    if step % 50 == 0:
        print(f"step {step:3d}  loss {loss.item():.4f}")
    if hit is None and loss.item() < 0.01:
        hit = step
        print(f"--> loss first below 0.01 at step {hit}")
        break
wandb.finish()
print("steps to loss < 0.01:", hit)

**Q4.2.** Sabotage it: with a **fresh, seeded** `LabNet` and the same batch, shuffle the labels (`yb[torch.randperm(len(yb))]`) and run the ritual again. The loss still goes to zero. What does that tell you about what the ritual does and does not test?

In [ ]:
# Q4.2 - same ritual, but with the labels shuffled so they no longer match.
seed_everything(0)
net = LabNet().to(DEV)
xb, yb = next(iter(train_loader))
xb, yb = xb.to(DEV), yb.to(DEV)
yb_shuffled = yb[torch.randperm(len(yb))]        # images unchanged, labels scrambled
opt = torch.optim.Adam(net.parameters(), lr=1e-3)

wandb.init(project="cv-lab2", name="ritual-shuffled-labels",
           config=dict(arch="LabNet", optimizer="Adam", lr=1e-3, steps=800,
                       batch=xb.size(0), params=count_params(net), seed=0,
                       labels="shuffled"))
net.train()
for step in range(800):
    opt.zero_grad()
    loss = F.cross_entropy(net(xb), yb_shuffled)
    loss.backward()
    opt.step()
    wandb.log({"train/loss": loss.item()}, step=step)
    if step % 100 == 0:
        print(f"step {step:3d}  loss {loss.item():.4f}")
print("final loss:", round(loss.item(), 4))
wandb.finish()

*Answer:* The loss still collapses to zero. So the ritual proves only that the machinery is wired correctly - forward pass, cross-entropy, backprop and the optimiser are connected, and the network has the capacity to **memorise** 128 images. It says nothing about **generalisation**: a model with this many parameters can fit an arbitrary, even meaningless, image-to-label mapping, so fitting one batch is no evidence it has learned anything transferable. The ritual is a wiring smoke-test; only held-out validation accuracy tells you the network actually learns the task.

## 5. Train for real, and read the curves

**Q5.1.** Seed, then train a fresh `LabNet` — keep it in a variable named `model`; Section 6 reads its weights — for 20 epochs with Adam at $10^{-3}$ — the lab's default optimiser: robust to a mediocre learning rate, at some cost in final accuracy. Config and curves to W&B as before. Plot training and validation accuracy per epoch from your returned history, and report the final test accuracy next to the MLP's.

In [ ]:
seed_everything(0)
model = LabNet().to(DEV)                          # keep as `model`; Section 6 reads it
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
config = dict(arch="LabNet", optimizer="Adam", lr=1e-3, epochs=20,
              params=count_params(model), seed=0)
hist = train(model, epochs=20, opt=opt, run_name="labnet-20ep-adam", config=config)

plt.figure(figsize=(6, 4))
plt.plot(hist["train/acc"], label="train")
plt.plot(hist["val/acc"], label="val")
plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.legend()
plt.title("LabNet - 20 epochs, Adam 1e-3")
plt.show()

print(f"LabNet final test acc: {hist['val/acc'][-1]:.3f}")
print(f"MLP    final test acc: {mlp_hist['val/acc'][-1]:.3f}")

**Q5.2.** Read your curves against the chapter's four regimes (the training-curves widget). Which shape is this? The book's reference run — the same network, SGD with momentum at lr 0.01, 40 epochs — reaches 77% validation accuracy. Where does yours land, and is the remaining train–validation gap overfitting or healthy?

*Answer:* This is the **healthy** regime: training and validation accuracy climb together with only a modest gap between them. With Adam at 1e-3 for 20 epochs the run lands in the **low-to-mid 70s%** on the test set - a few points under the book's 77% reference (which used SGD+momentum for twice as long). The remaining train-validation gap is **healthy, not overfitting**: the random crops and flips mean the network never sees the same image twice, so training accuracy stays pinned near validation instead of racing off to 100%. *(Replace with your run's exact final numbers.)*

**Q5.3.** Break it on purpose. Train a fresh, seeded `LabNet` — in a *separate* variable (say `bad`), so your Section 5 `model` survives for Section 6 — for 8 epochs with SGD at learning rate 1.0 (momentum 0.9) and log it. Which of the chapter's pathologies do your curves show, and what is the tell?

In [ ]:
seed_everything(0)
bad = LabNet().to(DEV)                            # separate variable; keep `model` intact
opt = torch.optim.SGD(bad.parameters(), lr=1.0, momentum=0.9)
config = dict(arch="LabNet", optimizer="SGD", lr=1.0, momentum=0.9, epochs=8,
              params=count_params(bad), seed=0)
bad_hist = train(bad, epochs=8, opt=opt, run_name="labnet-broken-lr1.0", config=config)

*Answer:* **Learning rate too high.** At lr 1.0 the SGD steps overshoot every minimum, so the loss diverges or bounces around (often to `nan`) instead of falling, and both accuracies sit at or near chance - about **10%** for ten classes. The tell is accuracy stuck around 0.10 with a loss that oscillates or explodes rather than decreasing: the optimiser is taking steps far too large to settle anywhere.

## 6. The bank, learned

Chapter 1 ended with a designed filter bank; Chapter 2's claim is that the first layer of your trained network has become one, with nobody choosing the numbers. Look.

**Q6.1.** Pull the first-layer weights from your Section 5 network (`model.c1.weight`, shape $32 \times 3 \times 3 \times 3$) and display all 32 filters as a grid of colour patches — normalise each filter to $[0,1]$ for display. At $3 \times 3$ they are hard to read; note what you can.

In [ ]:
# Q6.1 - first-layer filters of the Section 5 network, as colour patches.
w = model.c1.weight.detach().cpu()               # (32, 3, 3, 3)
fig, axes = plt.subplots(4, 8, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    f = w[i]
    f = (f - f.min()) / (f.max() - f.min() + 1e-8)   # normalise this filter to [0,1]
    ax.imshow(f.permute(1, 2, 0).numpy())
    ax.axis("off")
fig.suptitle("LabNet first-layer filters (3x3)")
plt.tight_layout()
plt.show()

**Q6.2.** Make them legible, the same way the book's filter figure does: widen only the first layer to $7 \times 7$ (`LabNet(first_kernel=7)`), retrain for 10 epochs (seeded, Adam, logged), and display the 32 filters again — expect structure, not a pixel match to the book's filter figure, which trained longer with a different optimiser. Beside them, plot a Gabor bank — Lab 1's `gabor_kernel` code, three frequencies at four orientations. What has the network rediscovered, and what does it have that the Gabor bank does not?

In [ ]:
# Q6.2 - widen only the first layer to 7x7, retrain, compare to a Gabor bank.
seed_everything(0)
model7 = LabNet(first_kernel=7).to(DEV)
opt = torch.optim.Adam(model7.parameters(), lr=1e-3)
config = dict(arch="LabNet-k7", optimizer="Adam", lr=1e-3, epochs=10,
              first_kernel=7, params=count_params(model7), seed=0)
hist7 = train(model7, epochs=10, opt=opt, run_name="labnet-k7-10ep", config=config)

# --- learned 7x7 first-layer filters ---
w = model7.c1.weight.detach().cpu()              # (32, 3, 7, 7)
fig, axes = plt.subplots(4, 8, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    f = w[i]
    f = (f - f.min()) / (f.max() - f.min() + 1e-8)
    ax.imshow(f.permute(1, 2, 0).numpy())
    ax.axis("off")
fig.suptitle("Learned first-layer filters (7x7)")
plt.tight_layout(); plt.show()

# --- Gabor bank: Lab 1's gabor_kernel, 3 frequencies x 4 orientations ---
def gabor_kernel(size=7, theta=0.0, freq=0.3, sigma=2.0):
    half = size // 2
    y, x = np.mgrid[-half:half + 1, -half:half + 1]
    x_t = x * np.cos(theta) + y * np.sin(theta)   # rotate coordinates
    envelope = np.exp(-(x**2 + y**2) / (2 * sigma**2))
    carrier = np.cos(2 * np.pi * freq * x_t)
    return envelope * carrier

freqs = [0.15, 0.30, 0.45]
thetas = [0, np.pi / 4, np.pi / 2, 3 * np.pi / 4]
fig, axes = plt.subplots(len(freqs), len(thetas), figsize=(6, 4.5))
for r, fr in enumerate(freqs):
    for c, th in enumerate(thetas):
        axes[r, c].imshow(gabor_kernel(7, theta=th, freq=fr), cmap="gray")
        axes[r, c].axis("off")
fig.suptitle("Gabor bank - 3 frequencies x 4 orientations")
plt.tight_layout(); plt.show()

*Answer:* Nothing designed them - the structure was driven entirely by gradient descent minimising cross-entropy on CIFAR-10, with backprop nudging the random initial weights toward whatever first-layer features most reduced the classification loss. Because natural images are made of oriented edges, bars and colour-opponent blobs, that optimisation pressure combined with the data's own statistics is what carved Gabor-like filters out of the initial noise. The 7x7 learned filters show the same vocabulary the Gabor bank is built from - oriented edges and bars at a range of angles and scales, plus blob / centre-surround shapes - so the network has rediscovered a Gabor-like filter bank from random initialisation, with nobody choosing the numbers. What it has that the Gabor bank does not is **colour**: because `c1` sees three input channels, many filters are colour-opponent (red-green, blue-yellow) rather than grayscale, and their exact frequencies and orientations are tuned to CIFAR-10's statistics rather than sampled on a fixed grid.

**Q6.3.** Nobody designed your filters. State, in two sentences, what did — where did the structure in those $7 \times 7$ patches come from?

*Answer:*

## 7. The record

The runs are the deliverable as much as the notebook. Paste links to your W&B runs below — they should include, at minimum: the MLP baseline, the one-batch ritual, the 20-epoch training run, the broken run, and the $7 \times 7$ retrain. Each must have a meaningful name and a complete config (architecture, optimiser, learning rate, epochs, parameter count, seed).

*W&B run links:*

## 8. Before you submit

- [ ] **Runtime ▸ Restart session and run all** on a GPU runtime — then read every output. The full re-run trains everything and takes roughly half an hour; budget for it.
- [ ] Group members filled in; every member can explain every cell — and has their own W&B account.
- [ ] Every *Answer:* cell answered; W&B links pasted in Section 7 and visible to a logged-out viewer or shared with the course staff.
- [ ] **File ▸ Download ▸ Download .ipynb**, one member submits on Moodle before **Monday 10 August, 09:00**.